# Ejercicio 1

Lo primero que toca hacer es crear el data set. Para hacer el data set se usa numpy para la generación y multiplicación de las matrices. Para las matrices se tiene esta notación:

$$A = \begin{pmatrix} a_{11} & a_{12} \\ a_{21} & a_{22} \end{pmatrix}, \quad B = \begin{pmatrix} b_{11} & b_{12} \\ b_{21} & b_{22} \end{pmatrix}$$

La matriz resultante es $C = A \times B$:

$$C = \begin{pmatrix} c_{11} & c_{12} \\ c_{21} & c_{22} \end{pmatrix}$$

Luego, para facilitar el entrenamiento, se aplanan las matrices a un dataset.

In [8]:
import numpy as np
import pandas as pd

np.random.seed(7)
num_muestras = 50000

A = np.random.randint(-20, 21, size=(num_muestras, 2, 2))
B = np.random.randint(-20, 21, size=(num_muestras, 2, 2))
C = np.matmul(A, B)

data = {
    'a11': A[:, 0, 0], 'a12': A[:, 0, 1],
    'a21': A[:, 1, 0], 'a22': A[:, 1, 1],
    'b11': B[:, 0, 0], 'b12': B[:, 0, 1],
    'b21': B[:, 1, 0], 'b22': B[:, 1, 1],
    'c11': C[:, 0, 0], 'c12': C[:, 0, 1],
    'c21': C[:, 1, 0], 'c22': C[:, 1, 1]
}

df = pd.DataFrame(data)
print(df.shape)
display(df.head())

(50000, 12)


,a11,a12,a21,a22,b11,b12,b21,b22,c11,c12,c21,c22
0,-16,5,-17,-1,-6,-3,-18,14,6,118,120,37
1,3,19,8,-6,9,-6,-15,-15,-258,-303,162,42
2,3,-12,5,6,9,4,-7,-14,111,180,3,-64
3,-12,19,18,-16,3,1,11,0,173,-12,-122,18
4,-13,-20,-9,-14,20,0,-5,-2,-160,40,-110,28


Ya con el modelo, simplemente se entrena el modelo. Para el modelo se usa un random forest de sklearn.

In [9]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error

X = df[['a11', 'a12', 'a21', 'a22', 'b11', 'b12', 'b21', 'b22']]
y = df[['c11', 'c12', 'c21', 'c22']]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = RandomForestRegressor(n_estimators=50, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
print(f"Error cuadrático medio (MSE) en prueba: {mse:.4f}")
print(f"Error absoluto medio (MAE) en prueba: {mae:.4f}")

Error cuadrático medio (MSE) en prueba: 8367.7412
Error absoluto medio (MAE) en prueba: 68.9116


Ya con el modelo, lo podemos evaluar con 10 ejemplos.

In [19]:
A_new = np.random.randint(-20, 21, size=(10, 2, 2))
B_new = np.random.randint(-20, 21, size=(10, 2, 2))
C_analytical = np.matmul(A_new, B_new)

X_new = pd.DataFrame({
    'a11': A_new[:, 0, 0], 'a12': A_new[:, 0, 1],
    'a21': A_new[:, 1, 0], 'a22': A_new[:, 1, 1],
    'b11': B_new[:, 0, 0], 'b12': B_new[:, 0, 1],
    'b21': B_new[:, 1, 0], 'b22': B_new[:, 1, 1]
})

predictions = model.predict(X_new)

for i in range(10):
    print(f"--- Ejemplo {i+1} ---\n")
    pred_matrix = predictions[i].reshape(2, 2)
    pred_rounded = np.round(pred_matrix).astype(int)

    print(f"{'Matriz A':<14} {'Matriz B':<14} {'Res. Analítico':<18} {'Predicción':<14}\n")

    a00, a01 = A_new[i][0]
    b00, b01 = B_new[i][0]
    c00, c01 = C_analytical[i][0]
    p00, p01 = pred_rounded[0]
    row1 = f"[{a00:>3} {a01:>3}]      [{b00:>3} {b01:>3}]      [{c00:>4} {c01:>4}]        [{p00:>4} {p01:>4}]"

    a10, a11 = A_new[i][1]
    b10, b11 = B_new[i][1]
    c10, c11 = C_analytical[i][1]
    p10, p11 = pred_rounded[1]
    row2 = f"[{a10:>3} {a11:>3}]      [{b10:>3} {b11:>3}]      [{c10:>4} {c11:>4}]        [{p10:>4} {p11:>4}]\n"

    print(row1)
    print(row2)

--- Ejemplo 1 ---

Matriz A       Matriz B       Res. Analítico     Predicción    

[ -2 -15]      [-16  12]      [-148  -84]        [-134  -60]
[-18   5]      [ 12   4]      [ 348 -196]        [ 196  -94]

--- Ejemplo 2 ---

Matriz A       Matriz B       Res. Analítico     Predicción    

[  8  -8]      [-15  15]      [-200  120]        [-124   92]
[-15  -1]      [ 10   0]      [ 215 -225]        [  94 -123]

--- Ejemplo 3 ---

Matriz A       Matriz B       Res. Analítico     Predicción    

[ -2  -6]      [-11  -6]      [  70    6]        [  39    2]
[-17   3]      [ -8   1]      [ 163  105]        [ 151   60]

--- Ejemplo 4 ---

Matriz A       Matriz B       Res. Analítico     Predicción    

[ -2  15]      [-11  -9]      [ 247  198]        [ 200  128]
[ -2 -13]      [ 15  12]      [-173 -138]        [-119 -103]

--- Ejemplo 5 ---

Matriz A       Matriz B       Res. Analítico     Predicción    

[  4   1]      [  2  13]      [  25   47]        [  41   10]
[-11  -9]      [ 17  -5]   

Se ve como los resultados del modelo son totalmente distintos a los resultados análiticos. Esto muestra que el modelo es muy poco apto para obtener los resultados de la multiplicación de matrices.

Por último, se evalua el rendimiento de los resultados analiticos y del modelo:

In [20]:
import time

num_tests = 100000
A_bench = np.random.randint(-20, 21, size=(num_tests, 2, 2))
B_bench = np.random.randint(-20, 21, size=(num_tests, 2, 2))

start_time = time.time()
C_bench = np.matmul(A_bench, B_bench)
analytical_time = time.time() - start_time

X_bench = pd.DataFrame({
    'a11': A_bench[:, 0, 0], 'a12': A_bench[:, 0, 1],
    'a21': A_bench[:, 1, 0], 'a22': A_bench[:, 1, 1],
    'b11': B_bench[:, 0, 0], 'b12': B_bench[:, 0, 1],
    'b21': B_bench[:, 1, 0], 'b22': B_bench[:, 1, 1]
})

start_time = time.time()
_ = model.predict(X_bench)
ml_time = time.time() - start_time

print(f"Método Analítico (Numpy): {analytical_time:.6f} segundos")
print(f"Método Machine Learning (Random Forest): {ml_time:.6f} segundos")
print(f"El modelo de ML es aproximadamente {ml_time / analytical_time:.2f} veces más lento que el cálculo analítico.")

Tiempo requerido para 100,000 multiplicaciones:
Método Analítico (Numpy): 0.003869 segundos
Método Machine Learning (Random Forest): 3.242553 segundos
El modelo de ML es aproximadamente 838.02 veces más lento que el cálculo analítico.


Se toman 100000 pruebas para medir los tiempos de ambos métodos. El cálculo analítico toma menos de 4 milisengundos, mientras que el modelo toma más de 3 segundos. Podemos concluir que el modelo de ML es menos preciso y mucho más lento. Por lo que para este problema específico, no es apropiado usar un modelo de ML.